# Interpretar un banco de flotación como respuesta cinética

**Pregunta:** ¿Por qué acortar un banco continuo cambia la recuperación de manera distinta de prolongar una prueba batch?

Nivel: 1. Requisitos previos: Ninguno; comienza aquí. Tiempo de lectura: 18 minutos.



## Objetivos

Ejecutar el rougher Cu-Mo y distinguir su curva batch virtual de la respuesta del banco alimentado de forma continua. Comprende por qué recuperación versus tiempo es una respuesta de proceso, no una animación decorativa.

## Preparación

```bash
pip install numpy==2.2.6 scipy==1.17.1 pandas==2.2.3 matplotlib==3.11.2 nbclient==0.11.0 nbformat==5.11.1 ipykernel==7.4.0 SALib==1.6.0 scikit-learn==1.5.2 ruff==0.15.18 pytest==9.1.1
```

El pipeline OreFlow se importa desde `data-pipeline/`; este notebook nunca instala paquetes.

Para la lección de nivel 1 sobre 02-flotation, registra las versiones fijadas antes de ejecutar el análisis; así se puede reconstruir el entorno que produjo cada tabla y figura.

In [ ]:
# Importar herramientas de análisis y registrar sus versiones exactas.
import importlib.metadata
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

repo = Path("../../../").resolve()
sys.path.insert(0, str(repo / "data-pipeline"))
for package in ("numpy", "scipy", "pandas", "matplotlib", "nbclient", "nbformat", "ipykernel"):
    print(package, importlib.metadata.version(package))
legend_style = dict(loc="lower center", bbox_to_anchor=(0.5, 1.02), frameon=False)


## Datos

Responsable: autores de OreFlow, licencia MIT. pipeline.cases.catalog aporta el circuito de cobre y molibdeno. El motor calcula alimentación rougher, tasas y residencia; tiempos batch y ajustes numéricos provienen de engine/data/constants.json. Los registros nominales, de banco corto y dosis modificada se generan cuando la lección los usa. Datos virtuales y referencia exacta comparten supuestos del simulador, sin ser mediciones de laboratorio. Fuente: https://github.com/fsantibanezleal/CAOS_OreFlow.

## Método

OreFlow agrega tasas por clase mediante $R_b(t)=\sum_j c_jx_j(1-e^{-k_jt})/\sum_j c_jx_j$. Un banco continuo tiene otra distribución temporal y se calcula por separado. Llamar a `simulate` y al constructor `trace` de producción.

Seleccionar un caso Cu-Mo de autoría propia y ejecutar los mismos métodos de circuito y trace del pipeline de datos OreFlow.

In [ ]:
# Seleccionar un caso Cu-Mo de autoría propia y ejecutar los mismos métodos de circuito y trace del
# pipeline de datos OreFlow.
from pipeline.cases.catalog import CASES
from pipeline.engine.circuit import simulate
from pipeline.engine.trace import trace

case = next((item for item in CASES if item.id == "copper_molybdenum"))
result = simulate(case.ore, case.plant, case.nominal)
record = trace(result, case.nominal, case.plant.family)
kinetics = record["methods"]["kinetics"]
print("familia:", record["family"], "| status:", kinetics["status"])
print("residencia del banco (min):", round(kinetics["bank"]["residence_min"], 2))
print("recuperación rougher del motor (%):", round(kinetics["bank"]["engine_rougher_pct"], 2))


Graficar la recuperación batch contra el tiempo y la recuperación del banco contra la posición de celda; los ejes x representan residencias distintas.

In [ ]:
# Graficar la recuperación batch contra el tiempo y la recuperación del banco contra la posición de
# celda; los ejes x representan residencias distintas.
plt.rcParams.update({"font.size": 22, "axes.titlepad": 8})
bank = record["curves"]["bank_profile"]
fig, axes = plt.subplots(2, 1, figsize=(6.4, 4.8))
axes[0].plot(kinetics["times_min"], kinetics["batch_recovery_pct"], "o-", label="Batch virtual")
axes[0].set(xlabel="Tiempo batch (min)", ylabel="Recuperación\n(%)", title="Respuesta batch")
axes[1].plot([v["cell"] for v in bank], [100 * v["recovery"] for v in bank], "o-", color="#b34d32")
axes[1].set(xlabel="Número de celda", ylabel="Recuperación\n(%)", title="Banco continuo")
for ax in axes:
    ax.set_ylim(0, 100)
    ax.grid(True, alpha=0.25)
fig.subplots_adjust(left=0.24, right=0.96, bottom=0.15, top=0.91, hspace=1.0)


Tiempo batch y posición de celda continua son coordenadas de exposición distintas. En batch cada partícula experimenta la duración indicada; el banco mezcla residencias y transfiere material entre etapas. Curvas parecidas no justifican igualar sus ejes horizontales. La tabla posterior conserva ley valiosa y sólidos para distinguir un cambio de recuperación de dilución o mayor colección de ganga.

Leer juntos alimentación rougher, concentrado y relave. Ley y recuperación responden preguntas distintas.

In [ ]:
# Leer juntos alimentación rougher, concentrado y relave.
streams = record["streams"]
rows = [
    (name, streams[name]["solids_tph"], streams[name]["grades"][kinetics["species"]])
    for name in ("rougher_feed", "rougher_concentrate", "rougher_tail")
    if name in streams
]
display(pd.DataFrame(rows, columns=["Flujo", "Sólidos (t/h)", f"{kinetics['species']} ley (%)"]))


Volver al rougher de cobre y molibdeno y generar un registro cinético nuevo desde sus tasas por clase. Cinco formas se ajustan a las mismas observaciones batch virtuales. La recuperación del banco usa residencia distribuida, no la exposición única del batch. El ajuste conecta ambas vistas sin tratar curvas como laboratorio. Flotación verdadera y recuperación rougher total pueden diferir porque el arrastre contribuye a la segunda; la referencia cinética aísla intencionalmente flotación verdadera.

In [ ]:
# Recalcular el rougher de cobre y molibdeno nominal y ajustar su respuesta batch virtual con la
# rutina cinética de producción.
from pipeline.cases.catalog import CASE_BY_ID
from pipeline.engine.circuit import simulate
from pipeline.engine.kinetics import kinetic_record

case = CASE_BY_ID["copper_molybdenum"]
nominal = simulate(case.ore, case.plant, case.nominal)
fits = kinetic_record(
    nominal.flotation,
    nominal.ore,
    case.nominal.rougher_cells,
    nominal.metrics["rougher_recovery_pct"],
)
display(
    pd.DataFrame(
        [
            {
                "Modelo": {
                    "first_order": "Primer orden",
                    "kelsall": "Kelsall",
                    "klimpel": "Klimpel",
                    "gamma": "Gamma",
                    "stretched_exponential": "Exponencial estirada",
                }.get(m["id"], m["id"]),
                "RMSE batch (pp)": m["rmse_pct"],
                "Error de banco (pp)": m["lumping_error_pct"],
            }
            for m in fits["models"]
        ]
    )
)


Retirar una celda continua y conservar alimentación y colector nominales. Disminuyen volumen de tanques instalado y residencia total para ese flujo. El motor calcula el banco corto en vez de truncar una curva batch. Inspeccionar recuperación antes de ajustar aproximaciones: una ecuación ajustada no entrega una referencia física independiente para su propia proyección. El cálculo es local a este banco y no implica que una celda aporte siempre la misma recuperación en otros circuitos.

In [ ]:
# Reducir en una la cantidad de celdas continuas sin cambiar alimentación ni dosis de reactivo.
short_point = case.nominal.with_values(rougher_cells=max(1, case.nominal.rougher_cells - 1))
short = simulate(case.ore, case.plant, short_point)
print("Recuperación del banco corto (%):", round(short.metrics["rougher_recovery_pct"], 3))


Ajustar el registro del banco corto e imprime la referencia exacta de flotación verdadera. Una partícula lenta puede recuperarse poco en la residencia disponible, aunque un batch prolongado alcance una recuperación última alta. Así se explica por qué extender tiempo batch y agregar una celda son intervenciones diferentes. Comparar referencias y formas conservando distinciones entre recuperación valiosa, recuperación de sólidos y ley. Se enseña residencia, no dimensionamiento de tanques de planta ni una cantidad recomendada de celdas.

In [ ]:
# Ajustar la misma familia cinética al banco acortado y proyectar cada ajuste sobre su geometría.
short_fits = kinetic_record(
    short.flotation, short.ore, short_point.rougher_cells, short.metrics["rougher_recovery_pct"]
)
print(
    "Recuperación exacta del banco corto (%):",
    round(short_fits["bank"]["exact_true_flotation_pct"], 3),
)


La referencia exacta excluye arrastre mecánico y la recuperación del flujo incluye todas las rutas de masa valiosa al concentrado. Pueden diferir sin error aritmético. Identificar especie pagable, base de residencia y si el agua lleva finos al concentrado. Una ecuación batch modela exposición de colección, pero no reconstruye por sí sola transferencias de agua y sólidos. Por eso se conserva la resolución de proceso junto al resumen cinético. Una auditoría de masa cerrada demuestra consistencia interna de esas transferencias, no calibración experimental de cada mecanismo.

## Resultados

**Respuesta:** Retirar una celda deja recuperación rougher global <!-- result:Recuperación global del banco corto (%) -->0<!-- /result --> por ciento y referencia de flotación verdadera <!-- result:Recuperación verdadera del banco corto (%) -->0<!-- /result --> por ciento. Exposición batch y residencia continua siguen siendo cálculos distintos; el arrastre separa además recuperación global y referencia cinética.

Resumir recuperación global y verdadera del banco corto junto a errores nominales para interpretar residencia.

In [ ]:
# Resumir las salidas calculadas que sostienen la interpretación de esta lección.
kinetic_by_name = {model["id"]: model for model in fits["models"]}
answer_values = {
    "Recuperación global del banco corto (%)": short.metrics["rougher_recovery_pct"],
    "Recuperación verdadera del banco corto (%)": short_fits["bank"]["exact_true_flotation_pct"],
    "RMSE batch de primer orden (pp)": kinetic_by_name["first_order"]["rmse_pct"],
    "RMSE batch de Kelsall (pp)": kinetic_by_name["kelsall"]["rmse_pct"],
    "Error de banco de primer orden (pp)": abs(kinetic_by_name["first_order"]["lumping_error_pct"]),
    "Error de banco de Kelsall (pp)": abs(kinetic_by_name["kelsall"]["lumping_error_pct"]),
}
answer_values = {label: round(float(number), 3) for label, number in answer_values.items()}
print(json.dumps(answer_values, ensure_ascii=False))


## Resumen

El solver calcula las respuestas batch y de banco del circuito simulado. El registro no es información de laboratorio, y recorrer la flowsheet no equivale al tiempo físico de planta.

## Referencias

- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3
- https://doi.org/10.1016/0301-7516(81)90019-3